In [ ]:
#import useful stuff                                                                                                                                                                                                             
import numpy as np
import matplotlib.pyplot as plt
import os

#check number of cores                                                                                                                                                                                                           
num_cores = os.cpu_count()
print(f"Number of CPU cores: {num_cores}")

N = 100000

In [ ]:
original_dist2neg = np.load("/oscar/data/mleblan6/SPECTER_hardprocess_dist2neg.npy", mmap_mode='r')

event_weight = np.load('/oscar/data/mleblan6/rjain/ppzjj_100k/weight_100k.npy')
neg_events = np.where(event_weight < 0)[0]

In [ ]:
np.shape(original_dist2neg)

In [ ]:
original_close2neg = np.zeros((len(neg_events), N))
for i in range(len(neg_events)):
    original_close2neg[i] = np.argsort(original_dist2neg[i])
    if i % 2000 == 0:
        print(i)

original_close2neg = original_close2neg[:, 1:]

In [ ]:
print(original_close2neg.shape)

In [ ]:
def cell_reweight(max_radius, verbose = False):
    reweighted_event_weight = np.copy(np.array(event_weight))
    cell_radius = []
    
    
    for i in range(N):  # search through all events
        final_cell_event = 'None'
        if reweighted_event_weight[i] < 0:  # if the i-th event has negative weight
            cell_weight = reweighted_event_weight[i]
            abs_cell_weight = abs(reweighted_event_weight[i])
            events_in_cell = [i]
    
            neg_event = np.where(np.array(neg_events) == i)[0].astype(int)[0]
    
              # search through all events
                
            for j in range(N-1):
                close2neg = original_close2neg[neg_event, j].astype(int)
                if (cell_weight <= 0.01) and (original_dist2neg[neg_event, close2neg] < max_radius):
    
                    cell_weight = np.append(cell_weight,
                        reweighted_event_weight[close2neg])
                    abs_cell_weight += abs(cell_weight[1])
                    cell_weight = np.sum(cell_weight)
                    events_in_cell.append(close2neg)
    
                    final_cell_event = close2neg

                else:
                    break
    
    
            if cell_weight > 0:
                cell_radius = np.append(
                    cell_radius,
                    original_dist2neg[neg_event, final_cell_event]
                )
                reweighted_event_weight[events_in_cell] = (
                    cell_weight / abs_cell_weight *
                    abs(reweighted_event_weight[events_in_cell])
                )
    
            if verbose == True:
                if i % 2000 == 0:
                    print(i)
    
    #print('Hard process reweighting completed')

    print(1 - len(np.where(reweighted_event_weight < 0)[0]) / len(neg_events),
          f'% of negative weights are reweighted for hard process events at {max_radius} GeV cell radius')

    return reweighted_event_weight


In [ ]:
semd = cell_reweight(1000)
print(1 - len(np.where(semd < 0)[0]) / len(neg_events))

In [ ]:
radii = np.logspace(2,5,50)
reweights = np.empty((0, N))

for i in radii:
    reweights = np.vstack((reweights, cell_reweight(i)))

In [ ]:
fracs = np.zeros(len(radii))

for i in range(len(radii)):
    fracs[i] = 1 - len(np.where(reweights[i,:] < 0)[0]) / len(neg_events)

In [ ]:
plt.plot(radii, fracs, marker = 'o', color = 'blue', linestyle = '--')
plt.axhline(y=1, color = 'black')
plt.xlabel('Max Cell Radius [GeV]')
plt.ylabel('Reweighted Fraction')
#plt.yscale('log')
plt.xscale('log')